In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU cache cleared.')
else:
    print('CUDA not available, skip GPU cache clear.')


# 下一课：Mini AlphaZero 核心循环

上一课你已经理解了 `MCTS` 的四个步骤。

这一课我们要回答 AlphaGo Zero / AlphaZero 里最关键的问题：

**神经网络和 MCTS 到底是怎么连起来的？**

这节课的重点不是做一个超强棋手，而是看懂 Mini AlphaZero 的核心训练循环：
- 网络输出 `policy` 和 `value`
- MCTS 用网络来引导搜索
- 搜索结果变成训练目标 `(state, pi, z)`
- 再反过来训练网络


## 1. AlphaZero 和纯 MCTS 的差别

上一课的 MCTS 里，扩展后主要靠随机 rollout 来估计局面价值。

但在 AlphaZero 里：
- 不再主要依赖随机 rollout
- 而是让神经网络直接给出：
  - `policy`：哪些动作更值得优先探索
  - `value`：当前局面对当前玩家大概有多好

所以你可以把它理解成：

**MCTS 负责搜索，神经网络负责提供先验和评估。**


## 2. `(state, pi, z)` 是什么

AlphaZero 训练时最经典的一条数据通常长这样：

- `state`：当前局面
- `pi`：MCTS 搜索后得到的动作分布
- `z`：最终胜负结果

含义分别是：
- `state`：网络输入
- `pi`：策略头训练目标
- `z`：价值头训练目标

所以网络不是直接拟合“人类下一手”，而是在拟合：
- 更强搜索给出的策略分布
- 最终整局胜负


In [ ]:
import math
import random
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 3. 继续使用井字棋，专心理解训练主线

围棋环境会复杂得多，所以这节课继续用井字棋。

这样你能把注意力全部放在：
- policy/value 网络
- MCTS 搜索统计
- 自我对弈数据生成
- 网络训练目标

等这条主线懂了，再换到围棋只是在状态和动作规模上放大。


In [ ]:
class TicTacToe:
    def __init__(self, auto_reset=True):
        if auto_reset:
            self.reset()

    def reset(self):
        self.board = [0] * 9
        self.current_player = 1
        return self.clone()

    def clone(self):
        env = TicTacToe(auto_reset=False)
        env.board = self.board[:]
        env.current_player = self.current_player
        return env

    def legal_actions(self):
        return [i for i, v in enumerate(self.board) if v == 0]

    def step(self, action):
        self.board[action] = self.current_player
        self.current_player *= -1

    def winner(self):
        lines = [
            (0, 1, 2), (3, 4, 5), (6, 7, 8),
            (0, 3, 6), (1, 4, 7), (2, 5, 8),
            (0, 4, 8), (2, 4, 6)
        ]
        for a, b, c in lines:
            s = self.board[a] + self.board[b] + self.board[c]
            if s == 3:
                return 1
            if s == -3:
                return -1
        return 0

    def is_terminal(self):
        return self.winner() != 0 or all(v != 0 for v in self.board)

    def result_for_player(self, player):
        w = self.winner()
        if w == player:
            return 1.0
        if w == 0:
            return 0.0
        return -1.0

    def encode(self):
        # 两个平面: 当前玩家子、对手子，再加一个当前行棋方平面
        current = np.array([1.0 if x == self.current_player else 0.0 for x in self.board], dtype=np.float32)
        opponent = np.array([1.0 if x == -self.current_player else 0.0 for x in self.board], dtype=np.float32)
        player_plane = np.full(9, 1.0 if self.current_player == 1 else 0.0, dtype=np.float32)
        return np.stack([current, opponent, player_plane], axis=0)

    def render(self):
        m = {1: 'X', -1: 'O', 0: '.'}
        return '\n'.join(' '.join(m[self.board[i + j]] for j in range(3)) for i in range(0, 9, 3))


## 4. 一个最小 policy-value 网络

这就是 AlphaZero 风格网络的最小原型：
- 输入当前局面
- 输出策略 logits
- 输出局面价值

你可以把它理解成“双头网络”：
- policy head 负责下一手倾向
- value head 负责局面优劣


In [ ]:
class PolicyValueNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Flatten(),
            nn.Linear(27, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        self.policy_head = nn.Linear(64, 9)
        self.value_head = nn.Linear(64, 1)

    def forward(self, x):
        h = self.backbone(x)
        policy_logits = self.policy_head(h)
        value = torch.tanh(self.value_head(h))
        return policy_logits, value.squeeze(1)


net = PolicyValueNet().to(device)
optimizer = optim.Adam(net.parameters(), lr=1e-3)


## 5. 用网络引导的 MCTS 节点

在 AlphaZero 风格 MCTS 里，节点不再只记录 visits / value，
还会记录每个动作的先验概率 `P`。

这些先验概率来自神经网络的 policy 输出。


In [ ]:
@dataclass
class AZNode:
    state: TicTacToe
    parent: 'AZNode' = None
    action_taken: int = None
    prior: float = 0.0
    children: dict = field(default_factory=dict)
    visits: int = 0
    value_sum: float = 0.0

    @property
    def q_value(self):
        return 0.0 if self.visits == 0 else self.value_sum / self.visits

    def expanded(self):
        return len(self.children) > 0


## 6. AlphaZero 风格选择公式

这次我们不用上一课的普通 UCT，而用一个更接近 AlphaZero 的思路：

$score = Q + U$

其中探索项大致依赖：
- 父节点访问次数
- 子节点访问次数
- 网络给出的先验概率 `prior`

意思就是：

**更有先验希望、但访问还不够多的动作，会被更积极探索。**


In [ ]:
def select_child(node, c_puct=1.5):
    best_score = -1e9
    best_child = None
    for child in node.children.values():
        u = c_puct * child.prior * math.sqrt(node.visits + 1) / (1 + child.visits)
        score = child.q_value + u
        if score > best_score:
            best_score = score
            best_child = child
    return best_child


def evaluate_state(net, state):
    x = torch.tensor(state.encode(), dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        logits, value = net(x)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    legal = state.legal_actions()
    masked = np.zeros_like(probs)
    masked[legal] = probs[legal]
    if masked.sum() == 0:
        masked[legal] = 1.0 / len(legal)
    else:
        masked /= masked.sum()
    return masked, float(value.item())


## 7. 用网络展开节点，不再随机 rollout

在这里，扩展节点后我们直接用网络做两件事：
- 给合法动作分配先验概率
- 给当前节点估一个 value

然后这个 value 会被回传上去。

这就是 AlphaZero 和普通随机 rollout MCTS 的关键差别之一。


In [ ]:
def expand_and_evaluate(node, net):
    priors, value = evaluate_state(net, node.state)
    for action in node.state.legal_actions():
        next_state = node.state.clone()
        next_state.step(action)
        node.children[action] = AZNode(
            state=next_state,
            parent=node,
            action_taken=action,
            prior=float(priors[action])
        )
    return value


def backpropagate(node, value):
    current = node
    while current is not None:
        current.visits += 1
        current.value_sum += value
        value = -value
        current = current.parent


In [ ]:
def run_az_mcts(root_state, net, num_simulations=50):
    root = AZNode(state=root_state.clone())
    root_value = expand_and_evaluate(root, net)
    root.visits += 1
    root.value_sum += root_value

    for _ in range(num_simulations):
        node = root

        while node.expanded() and not node.state.is_terminal():
            node = select_child(node)

        if node.state.is_terminal():
            value = node.state.result_for_player(node.state.current_player)
        else:
            value = expand_and_evaluate(node, net)

        backpropagate(node, value)

    return root


def visit_count_policy(root):
    pi = np.zeros(9, dtype=np.float32)
    for action, child in root.children.items():
        pi[action] = child.visits
    if pi.sum() > 0:
        pi /= pi.sum()
    return pi


## 8. 先看一条 `(state, pi, z)` 数据长什么样

下面我们先只跑一步 MCTS，看看：
- 当前状态是什么
- 搜索后的动作分布 `pi` 是什么

`z` 要等整局结束后才知道，所以后面自我对弈时再一起生成。


In [ ]:
env = TicTacToe()
root = run_az_mcts(env, net, num_simulations=80)
pi = visit_count_policy(root)

print('当前状态：')
print(env.render())
print()
print('搜索后策略分布 pi：')
print(np.round(pi, 3))
print('推荐动作:', int(np.argmax(pi)))


## 9. 自我对弈如何生成 `(state, pi, z)`

这里就是 Mini AlphaZero 最关键的数据生成部分：

每一步：
- 保存当前状态 `state`
- 跑 MCTS 拿到搜索分布 `pi`
- 按 `pi` 选动作继续下

等整局结束后，再把最终结果 `z` 回填给整条轨迹里的每一步。


In [ ]:
def self_play_game(net, num_simulations=60):
    env = TicTacToe()
    history = []

    while not env.is_terminal():
        state_copy = env.clone()
        player = env.current_player

        root = run_az_mcts(env, net, num_simulations=num_simulations)
        pi = visit_count_policy(root)

        history.append((state_copy.encode(), pi, player))

        action = int(np.argmax(pi))
        env.step(action)

    winner = env.winner()
    data = []
    for state, pi, player in history:
        if winner == 0:
            z = 0.0
        elif winner == player:
            z = 1.0
        else:
            z = -1.0
        data.append((state, pi, z))
    return data, env, winner


game_data, final_env, winner = self_play_game(net, num_simulations=40)
print('最终棋盘：')
print(final_env.render())
print('winner:', winner)
print('生成样本数:', len(game_data))
print('第一条样本的 z:', game_data[0][2])


## 10. 网络是怎么训练的

现在我们已经有了 `(state, pi, z)`，训练就很清楚了：

- 策略头去拟合 `pi`
- 价值头去拟合 `z`

也就是说：
- policy 学“搜索后的更强落子分布”
- value 学“最终胜负结果”


In [ ]:
states = torch.tensor(np.array([x[0] for x in game_data]), dtype=torch.float32, device=device)
target_pis = torch.tensor(np.array([x[1] for x in game_data]), dtype=torch.float32, device=device)
target_zs = torch.tensor(np.array([x[2] for x in game_data]), dtype=torch.float32, device=device)

policy_logits, pred_values = net(states)

# policy loss: 拟合搜索后的策略分布 pi
log_probs = torch.log_softmax(policy_logits, dim=1)
policy_loss = -(target_pis * log_probs).sum(dim=1).mean()

# value loss: 拟合最终胜负 z
value_loss = nn.functional.mse_loss(pred_values, target_zs)

loss = policy_loss + value_loss

optimizer.zero_grad()
loss.backward()
optimizer.step()

print('policy_loss:', round(float(policy_loss.item()), 4))
print('value_loss:', round(float(value_loss.item()), 4))
print('total_loss:', round(float(loss.item()), 4))


## 11. 把整个 Mini AlphaZero 主线串起来

你现在可以把这节课总结成一个循环：

1. 当前网络给出 policy/value
2. MCTS 利用 policy/value 做搜索
3. 自我对弈产生 `(state, pi, z)`
4. 用这些数据训练网络
5. 新网络再去做更强的搜索和自我对弈

这就是 AlphaZero 的核心闭环。


## 12. 这节课最该记住什么

如果你想抓住 Mini AlphaZero 的本质，就记住这句话：

**网络不是直接模仿人类落子，而是在模仿“搜索后的更强策略”，并同时学习最终胜负。**

这和普通监督学习下棋最大的区别就在这里。


## 13. 下一课最自然学什么

学完这节后，最自然的下一步有两个：
- 把这套 Mini AlphaZero 逻辑扩成一个完整可迭代训练脚本
- 或者开始做一个更像围棋的小棋盘环境

如果按你的兴趣，我建议下一课直接做：

**Mini AlphaZero for 5x5 Go 的环境设计。**
